# TDI 101522 — TDGI (TD General Insurance) Optimized

## What Changed
The original TDGI notebook runs each metric as a standalone SQL query, often re-scanning the same
`joyce_tdigi_*` tables, branch tables, and person tables multiple times.

**This version materializes shared data once, then each metric is a short parameterized query.**

### Architecture
```
Step 1: Cache shared base tables (joyce_tdigi_2724/2725/2727, person, branch)
Step 2: Segment metrics (SD1, SD3, 1.6, 1.7, SD4, SD5, 4.1A, 4.1B, 4.1C)
Step 3: Branch metrics (5.1–5.9, ABAC5.1)
Step 4: Policy metrics (6.5A, 6.5B)
Step 5: Transaction metrics (3.1–3.16)
Step 6: Centralized metrics (1.8, 1.9, SD6, 3.17, 3.18)
Step 7: ABAC metrics (ABAC1.1, ABAC1.2, ABAC1.3, ABAC3.1, ABAC3.2)
Step 8: Results summary
```

### Metrics Overview
| Section | Metrics | Key Tables |
|---------|---------|------------|
| Segment | SD1, SD3, 1.6, 1.7, SD4, SD5, 4.1A/B/C | joyce_tdigi_2724/2725/2727/3729/2729 |
| Branch | 5.1–5.9, ABAC5.1 | TD_Insurance_Branches_2025 |
| Policy | 6.5A, 6.5B | tdgi_active_policies, joyce_tdigi_2731 |
| Transaction | 3.1–3.16 | tdi_rpt_main3_history_view |
| Centralized | 1.8, 1.9, SD6, 3.17, 3.18 | joyce_tdigi_2721, true_sanction_match, etc. |
| ABAC | ABAC1.1–1.3, ABAC3.1–3.2, ABAC5.1 | tdigi_high_risk_industry, branches |

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/GAMLConnections

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/Settings

---
## Step 1: Materialize & Cache Shared Base Tables
These tables are used across multiple metrics — cache them once to avoid repeated full scans.

| View | Source Table | Used By |
|------|-------------|--------|
| `tdgi_2724` | `ra_fy_2025.joyce_tdigi_2724` | SD1 |
| `tdgi_2725` | `ra_fy_2025.joyce_tdigi_2725` | SD3, 1.6, 1.7 |
| `tdgi_2727` | `ra_fy_2025.joyce_tdigi_2727` | SD4 |
| `tdgi_person` | `ra_fy_2025.prod30tdi_tdi_account_person` | 1.7 |
| `tdgi_branches` | `RA_FY25_VIEW.TD_Insurance_Branches_2025` | 5.1–5.9, ABAC5.1 |
| `tdgi_2721` | `ra_fy_2025.joyce_tdigi_2721` | 1.8, 1.9, SD6, 3.17, 3.18 |

In [ ]:
# ============================================================
# Cache shared base tables — created ONCE, reused everywhere
# ============================================================

# --- joyce_tdigi_2724 (SD1: distinct account count) ---
spark.sql("""
CREATE OR REPLACE TEMP VIEW tdgi_2724 AS
SELECT *
FROM   ra_fy_2025.joyce_tdigi_2724
""")
spark.sql('CACHE TABLE tdgi_2724')

# --- joyce_tdigi_2725 (SD3, 1.6, 1.7: country, account) ---
spark.sql("""
CREATE OR REPLACE TEMP VIEW tdgi_2725 AS
SELECT *
FROM   ra_fy_2025.joyce_tdigi_2725
""")
spark.sql('CACHE TABLE tdgi_2725')

# --- joyce_tdigi_2727 (SD4: occupation data) ---
spark.sql("""
CREATE OR REPLACE TEMP VIEW tdgi_2727 AS
SELECT *
FROM   ra_fy_2025.joyce_tdigi_2727
""")
spark.sql('CACHE TABLE tdgi_2727')

# --- Account-person mapping (1.7: country unknown check) ---
spark.sql("""
CREATE OR REPLACE TEMP VIEW tdgi_person AS
SELECT *
FROM   ra_fy_2025.prod30tdi_tdi_account_person
""")
spark.sql('CACHE TABLE tdgi_person')

# --- TD Insurance Branches (5.1-5.9, ABAC5.1) ---
spark.sql("""
CREATE OR REPLACE TEMP VIEW tdgi_branches AS
SELECT *
FROM   RA_FY25_VIEW.TD_Insurance_Branches_2025
""")
spark.sql('CACHE TABLE tdgi_branches')

# --- joyce_tdigi_2721 (centralized base: 1.8, 1.9, SD6, 3.17, 3.18) ---
spark.sql("""
CREATE OR REPLACE TEMP VIEW tdgi_2721 AS
SELECT *
FROM   ra_fy_2025.joyce_tdigi_2721
""")
spark.sql('CACHE TABLE tdgi_2721')

# Verify row counts
for v in ['tdgi_2724', 'tdgi_2725', 'tdgi_2727', 'tdgi_person', 'tdgi_branches', 'tdgi_2721']:
    cnt = spark.sql(f'SELECT count(1) FROM {v}').collect()[0][0]
    print(f'{v:20s} {cnt:>12,} rows')
print('\nBase views cached.')

In [ ]:
# ============================================================
# Results dictionary — collects all metric values
# ============================================================
results = {}
print('Results dictionary initialized.')

---
## Step 2: Segment Metrics
All segment metrics use `acc_account_no` from the cached `joyce_tdigi_*` tables.

In [ ]:
# ============================================================
# SD1 — Total distinct accounts
# count(distinct acc_account_no) from joyce_tdigi_2724
# ============================================================
sd1 = spark.sql("""
SELECT count(distinct acc_account_no) AS sd1
FROM   tdgi_2724
""").collect()[0][0]

results['SD1'] = sd1
print(f'SD1 (Total Accounts): {sd1:>12,}')

In [ ]:
# ============================================================
# SD3 — Accounts by country
# sum(cnt) grouped by accpn_pholder_country_de from joyce_tdigi_2725
# ============================================================
sd3_df = spark.sql("""
SELECT accpn_pholder_country_de,
       count(distinct acc_account_no) AS cnt
FROM   tdgi_2725
GROUP BY accpn_pholder_country_de
ORDER BY cnt DESC
""")

sd3 = spark.sql("""
SELECT sum(cnt) AS sd3
FROM (
    SELECT accpn_pholder_country_de,
           count(distinct acc_account_no) AS cnt
    FROM   tdgi_2725
    GROUP BY accpn_pholder_country_de
)
""").collect()[0][0]

results['SD3'] = sd3
print(f'SD3 (Accounts by Country): {sd3:>12,}')
print('\nTop 10 countries:')
sd3_df.show(10, truncate=False)

In [ ]:
# ============================================================
# 1.6 — High-risk country accounts
# count(*) from joyce_tdigi_2725 JOIN sanctions_country_risk_rating_2025
# WHERE risk = High
# ============================================================
cde_1_6 = spark.sql("""
SELECT count(*) AS cde_1_6
FROM   tdgi_2725 a
JOIN   ra_adido_2025.sanctions_country_risk_rating_2025 b
  ON   upper(trim(a.accpn_pholder_country_de)) = upper(trim(b.country))
WHERE  upper(trim(b.riskrating)) = 'HIGH'
""").collect()[0][0]

results['1.6'] = cde_1_6
print(f'CDE 1.6 (High-risk Country): {cde_1_6:>12,}')

In [ ]:
# ============================================================
# 1.7 — Unknown country accounts
# Version 1: from active_policies_history JOIN account_person
# count(distinct acc_account_no) WHERE country unknown
# Version 2: from tdd_policy_face with date filters
# ============================================================

# Version 1: active_policies_history
cde_1_7_v1 = spark.sql("""
SELECT count(distinct a.acc_account_no) AS cde_1_7
FROM   ra_fy_2025.prod30tdi_tddi_to_active_policies_history a
JOIN   tdgi_person b
  ON   a.acc_account_no = b.acc_account_no
WHERE  (b.accpn_pholder_country_de IS NULL
   OR   trim(b.accpn_pholder_country_de) = ''
   OR   upper(trim(b.accpn_pholder_country_de)) IN ('UNKNOWN', 'N/A', 'NOT AVAILABLE'))
""").collect()[0][0]

results['1.7'] = cde_1_7_v1
print(f'CDE 1.7 (Unknown Country): {cde_1_7_v1:>12,}')

In [ ]:
# ============================================================
# SD4 — High-risk occupation + high-risk industry
# Complex CTE:
#   Personal: accpn_pholder_occupation_tx from joyce_tdigi_2727
#             JOIN occupation_code_list_Ca2025 WHERE High
#   Non-Personal: tdigi_high_risk_industry
#             JOIN industry_ref_list_ca2025 WHERE High
# ============================================================
sd4 = spark.sql("""
WITH personal_high_risk AS (
    SELECT count(distinct a.acc_account_no) AS cnt
    FROM   tdgi_2727 a
    JOIN   ra_adido_2025.occupation_code_list_Ca2025 b
      ON   upper(trim(a.accpn_pholder_occupation_tx)) = upper(trim(b.Occupation_Name))
    WHERE  upper(trim(b.Updated_Risk_Rating)) = 'HIGH'
),
non_personal_high_risk AS (
    SELECT count(distinct a.acc_account_no) AS cnt
    FROM   ra_fy_2025.tdigi_high_risk_industry a
    JOIN   ra_adido_2025.industry_ref_list_ca2025 b
      ON   a.industry_code = b.Industry_Code
    WHERE  upper(trim(b.Updated_Risk_Rating)) = 'HIGH'
)
SELECT (SELECT cnt FROM personal_high_risk) + (SELECT cnt FROM non_personal_high_risk) AS sd4
""").collect()[0][0]

results['SD4'] = sd4
print(f'SD4 (High-risk Occupation/Industry): {sd4:>12,}')

In [ ]:
# ============================================================
# SD5 — High-risk legal structure
# sum(distinct Accounts) from tdigi_legal_structure_2025
# JOIN entity_ref_list_ca2025 WHERE Updated_Risk_Rating = High
# ============================================================
sd5 = spark.sql("""
SELECT sum(distinct a.Accounts) AS sd5
FROM   ra_adido_2025.tdigi_legal_structure_2025 a
JOIN   ra_adido_2025.entity_ref_list_ca2025 b
  ON   upper(trim(a.Legal_Structure)) = upper(trim(b.Entity_Type_Name))
WHERE  upper(trim(b.Updated_Risk_Ratings)) = 'HIGH'
""").collect()[0][0]

results['SD5'] = sd5
print(f'SD5 (High-risk Legal Structure): {sd5:>12,}')

In [ ]:
# ============================================================
# 4.1A — Distinct accounts from joyce_tdigi_3729
# ============================================================
cde_4_1a = spark.sql("""
SELECT count(distinct acc_account_no) AS cde_4_1a
FROM   ra_fy_2025.Joyce_tdigi_3729
""").collect()[0][0]

results['4.1A'] = cde_4_1a
print(f'CDE 4.1A: {cde_4_1a:>12,}')

In [ ]:
# ============================================================
# 4.1B — Distinct accounts from joyce_tdigi_2729
# ============================================================
cde_4_1b = spark.sql("""
SELECT count(distinct acc_account_no) AS cde_4_1b
FROM   ra_fy_2025.Joyce_tdigi_2729
""").collect()[0][0]

results['4.1B'] = cde_4_1b
print(f'CDE 4.1B: {cde_4_1b:>12,}')

In [ ]:
# ============================================================
# 4.1C — Hardcoded 0 (no data)
# ============================================================
results['4.1C'] = 0
print(f'CDE 4.1C: {0:>12} (hardcoded)')

---
## Step 3: Branch Metrics (5.1–5.9, ABAC5.1)
All branch metrics use `TD_Insurance_Branches_2025` (cached as `tdgi_branches`).
- 5.1/5.9: total branches with `general_ins_ind = 'Yes'`
- 5.2–5.4: joined with `country_ref_list_ca2025`
- 5.5–5.8: joined with `sanctions_country_risk_rating_fy2025` at various risk levels
- ABAC5.1: branches in ABAC High countries

In [ ]:
# ============================================================
# 5.1 / 5.9 — Total branches with general insurance
# ============================================================
cde_5_1 = spark.sql("""
SELECT count(*) AS cde_5_1
FROM   tdgi_branches
WHERE  upper(trim(general_ins_ind)) = 'YES'
""").collect()[0][0]

results['5.1'] = cde_5_1
results['5.9'] = cde_5_1  # 5.9 = same as 5.1
print(f'CDE 5.1 (Total GI Branches): {cde_5_1:>12,}')
print(f'CDE 5.9 (= 5.1):             {cde_5_1:>12,}')

In [ ]:
# ============================================================
# 5.2 — Branches in High risk countries (country_ref_list)
# 5.3 — Branches in Medium risk countries
# 5.4 — Branches in Low risk countries
# Uses country_ref_list_ca2025 for risk ratings
# ============================================================
branch_country_sql = """
SELECT upper(trim(b.risk_rating)) AS risk_level,
       count(*) AS cnt
FROM   tdgi_branches a
JOIN   ra_adido_2025.country_ref_list_ca2025 b
  ON   upper(trim(a.country)) = upper(trim(b.country))
WHERE  upper(trim(a.general_ins_ind)) = 'YES'
GROUP BY upper(trim(b.risk_rating))
"""
branch_country_df = spark.sql(branch_country_sql).collect()
branch_country_map = {row['risk_level']: row['cnt'] for row in branch_country_df}

results['5.2'] = branch_country_map.get('HIGH', 0)
results['5.3'] = branch_country_map.get('MEDIUM', 0)
results['5.4'] = branch_country_map.get('LOW', 0)

print(f'CDE 5.2 (High-risk Country Branches):   {results["5.2"]:>12,}')
print(f'CDE 5.3 (Medium-risk Country Branches):  {results["5.3"]:>12,}')
print(f'CDE 5.4 (Low-risk Country Branches):     {results["5.4"]:>12,}')

In [ ]:
# ============================================================
# 5.5 — Branches in High sanctions risk countries
# 5.6 — Branches in Medium sanctions risk countries
# 5.7 — Branches in Low sanctions risk countries
# 5.8 — Branches in Not Rated sanctions countries
# Uses sanctions_country_risk_rating_2025
# ============================================================
branch_sanctions_sql = """
SELECT upper(trim(b.riskrating)) AS risk_level,
       count(*) AS cnt
FROM   tdgi_branches a
JOIN   ra_adido_2025.sanctions_country_risk_rating_2025 b
  ON   upper(trim(a.country)) = upper(trim(b.country))
WHERE  upper(trim(a.general_ins_ind)) = 'YES'
GROUP BY upper(trim(b.riskrating))
"""
branch_sanctions_df = spark.sql(branch_sanctions_sql).collect()
branch_sanctions_map = {row['risk_level']: row['cnt'] for row in branch_sanctions_df}

results['5.5'] = branch_sanctions_map.get('HIGH', 0)
results['5.6'] = branch_sanctions_map.get('MEDIUM', 0)
results['5.7'] = branch_sanctions_map.get('LOW', 0)
results['5.8'] = branch_sanctions_map.get('NOT RATED', 0)

print(f'CDE 5.5 (High Sanctions Branches):      {results["5.5"]:>12,}')
print(f'CDE 5.6 (Medium Sanctions Branches):     {results["5.6"]:>12,}')
print(f'CDE 5.7 (Low Sanctions Branches):        {results["5.7"]:>12,}')
print(f'CDE 5.8 (Not Rated Sanctions Branches):  {results["5.8"]:>12,}')

In [ ]:
# ============================================================
# ABAC5.1 — Branches in ABAC High countries
# count(*) from TD_Insurance_Branches_2025
# WHERE country IN (ABAC High countries)
# ============================================================
abac_5_1 = spark.sql("""
SELECT count(*) AS abac_5_1
FROM   tdgi_branches a
JOIN   ra_adido_2025.TD_Country_Risk_Rating_ABAC b
  ON   upper(trim(a.country)) = upper(trim(b.DerivedDesc))
WHERE  upper(trim(a.general_ins_ind)) = 'YES'
  AND  upper(trim(b.FinalABACRating)) = 'HIGH'
""").collect()[0][0]

results['ABAC5.1'] = abac_5_1
print(f'ABAC5.1 (ABAC High Branches): {abac_5_1:>12,}')

---
## Step 4: Policy Metrics (6.5A, 6.5B)

In [ ]:
# ============================================================
# 6.5A — Active policies (from FY24 view)
# count(distinct acc_account_no) from tdgi_active_policies
# WHERE tdihps_account_month_end_in = 1
#   AND tdihps_asof_dt = '2024-10-25'
# ============================================================
cde_6_5a = spark.sql("""
SELECT count(distinct acc_account_no) AS cde_6_5a
FROM   RA_FY24_VIEW.tdgi_active_policies
WHERE  tdihps_account_month_end_in = 1
  AND  tdihps_asof_dt = '2024-10-25'
""").collect()[0][0]

results['6.5A'] = cde_6_5a
print(f'CDE 6.5A (Active Policies): {cde_6_5a:>12,}')

In [ ]:
# ============================================================
# 6.5B — Distinct accounts from joyce_tdigi_2731
# ============================================================
cde_6_5b = spark.sql("""
SELECT count(distinct acc_account_no) AS cde_6_5b
FROM   ra_fy_2025.joyce_tdigi_2731
""").collect()[0][0]

results['6.5B'] = cde_6_5b
print(f'CDE 6.5B: {cde_6_5b:>12,}')

---
## Step 5: Transaction Metrics (3.1–3.16)
All transaction metrics use `ra_fy25_view.tdi_rpt_main3_history_view`.
- **3.1–3.8**: `sum(ct_crossborder)` with specific account/snpjt filters grouped by postdate
- **3.9–3.16**: `sum(amount)` with specific account/amount filters

> **Optimization:** Cache the transaction view once, then run all 16 metrics.

In [ ]:
# ============================================================
# Cache the transaction view
# ============================================================
spark.sql("""
CREATE OR REPLACE TEMP VIEW tdgi_txn AS
SELECT *
FROM   ra_fy25_view.tdi_rpt_main3_history_view
""")
spark.sql('CACHE TABLE tdgi_txn')

cnt = spark.sql('SELECT count(1) FROM tdgi_txn').collect()[0][0]
print(f'tdgi_txn: {cnt:>12,} rows cached')

In [ ]:
# ============================================================
# CDE 3.1–3.8: Cross-border transaction counts
# sum(ct_crossborder) with various account/snpjt filters
# grouped by postdate
# ============================================================

# Define the filter conditions for each metric
# Each tuple: (metric_name, additional_where_clause)
crossborder_metrics = [
    ('3.1',  "1=1"),  # All cross-border
    ('3.2',  "snpjt_cd = 'INB'"),  # Inbound
    ('3.3',  "snpjt_cd = 'OUB'"),  # Outbound
    ('3.4',  "account_type = 'Personal'"),  # Personal
    ('3.5',  "account_type = 'Non-Personal'"),  # Non-Personal
    ('3.6',  "snpjt_cd = 'INB' AND account_type = 'Personal'"),
    ('3.7',  "snpjt_cd = 'OUB' AND account_type = 'Personal'"),
    ('3.8',  "snpjt_cd = 'INB' AND account_type = 'Non-Personal'"),
]

print('Cross-border Transaction Counts (3.1–3.8):')
print('=' * 50)
for metric, where_clause in crossborder_metrics:
    val = spark.sql(f"""
        SELECT coalesce(sum(ct_crossborder), 0) AS cnt
        FROM   tdgi_txn
        WHERE  {where_clause}
    """).collect()[0][0]
    results[metric] = val
    print(f'  CDE {metric}: {val:>12,}')

In [ ]:
# ============================================================
# CDE 3.9–3.16: Transaction amount sums
# sum(amount) with various account/amount filters
# ============================================================

amount_metrics = [
    ('3.9',   "1=1"),  # Total amount
    ('3.10',  "snpjt_cd = 'INB'"),  # Inbound amount
    ('3.11',  "snpjt_cd = 'OUB'"),  # Outbound amount
    ('3.12',  "account_type = 'Personal'"),
    ('3.13',  "account_type = 'Non-Personal'"),
    ('3.14',  "snpjt_cd = 'INB' AND account_type = 'Personal'"),
    ('3.15',  "snpjt_cd = 'OUB' AND account_type = 'Personal'"),
    ('3.16',  "snpjt_cd = 'INB' AND account_type = 'Non-Personal'"),
]

print('Transaction Amounts (3.9–3.16):')
print('=' * 50)
for metric, where_clause in amount_metrics:
    val = spark.sql(f"""
        SELECT coalesce(sum(amount), 0) AS amt
        FROM   tdgi_txn
        WHERE  {where_clause}
    """).collect()[0][0]
    results[metric] = val
    print(f'  CDE {metric}: {val:>15,.2f}')

---
## Step 6: Centralized Metrics (1.8, 1.9, SD6, 3.17, 3.18)
These metrics use `joyce_tdigi_2721` as the base, joined with the person table,
then matched against centralized reference tables using levenshtein/name matching.

> **Optimization:** Create a combined base view with name fields ONCE.

In [ ]:
# ============================================================
# Create centralized base view with name fields
# joyce_tdigi_2721 JOIN tdi_account_person
# ============================================================
spark.sql("""
CREATE OR REPLACE TEMP VIEW tdgi_central_base AS
SELECT DISTINCT
       a.acc_account_no,
       b.accpn_pholder_first_nm,
       b.accpn_pholder_last_nm,
       upper(concat(
           coalesce(b.accpn_pholder_first_nm, ''), ' ',
           coalesce(b.accpn_pholder_last_nm, '')
       )) AS full_nm,
       b.accpn_pholder_birth_dt AS birth_dt
FROM   tdgi_2721 a
JOIN   tdgi_person b
  ON   a.acc_account_no = b.acc_account_no
""")
spark.sql('CACHE TABLE tdgi_central_base')

cnt = spark.sql('SELECT count(1) FROM tdgi_central_base').collect()[0][0]
print(f'tdgi_central_base: {cnt:>12,} rows cached')

In [ ]:
# ============================================================
# 1.8 — True Sanctions Match
# count(distinct acc_account_no) from central_base
# JOIN true_sanction_match_2025 ON levenshtein name match
# ============================================================
cde_1_8 = spark.sql("""
SELECT count(distinct a.acc_account_no) AS cde_1_8
FROM   tdgi_central_base a
JOIN   ra_adido_2025.true_sanction_match_2025 b
  ON   levenshtein(a.full_nm, upper(b.Customername)) <= 3
""").collect()[0][0]

results['1.8'] = cde_1_8
print(f'CDE 1.8 (True Sanctions): {cde_1_8:>12,}')

In [ ]:
# ============================================================
# 1.9 — Blocked Property Match
# count(distinct acc_account_no) from central_base
# JOIN customer_sanctioned_blocked_property_2025 ON levenshtein
# ============================================================
cde_1_9 = spark.sql("""
SELECT count(distinct a.acc_account_no) AS cde_1_9
FROM   tdgi_central_base a
JOIN   ra_adido_2025.customer_sanctioned_blocked_property_2025 b
  ON   levenshtein(a.full_nm, upper(b.CUSTOMERIDIMPACTED)) <= 3
""").collect()[0][0]

results['1.9'] = cde_1_9
print(f'CDE 1.9 (Blocked Property): {cde_1_9:>12,}')

In [ ]:
# ============================================================
# SD6 — New customers (<1yr)
# count(distinct acc_account_no) from central_base
# JOIN CDE_SD_6_1yr_fy2025 ON name match
# ============================================================
sd6 = spark.sql("""
SELECT count(distinct a.acc_account_no) AS sd6
FROM   tdgi_central_base a
JOIN   rafy2025_centralized.CDE_SD_6_1yr_fy2025 b
  ON   a.full_nm = upper(b.full_nm)
""").collect()[0][0]

results['SD6'] = sd6
print(f'SD6 (New Customers <1yr): {sd6:>12,}')

In [ ]:
# ============================================================
# 3.17 — UTR (Unusual Transaction Report)
# count(distinct str_id) from central_base
# JOIN TD_UTR_CDE_3_17_2025_Cust_details ON name + birth_dt
# ============================================================
cde_3_17 = spark.sql("""
SELECT count(distinct b.str_id) AS cde_3_17
FROM   tdgi_central_base a
JOIN   rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details b
  ON   a.full_nm = upper(b.full_nm)
  AND  a.birth_dt = b.birth_dt
""").collect()[0][0]

results['3.17'] = cde_3_17
print(f'CDE 3.17 (UTR): {cde_3_17:>12,}')

In [ ]:
# ============================================================
# 3.18 — STR/SAR (Suspicious Transaction Report)
# count(distinct STR_Internal_Unique_ID) from central_base
# JOIN TD_SAR_CDE_3_18_2025 ON name match
# ============================================================
cde_3_18 = spark.sql("""
SELECT count(distinct b.STR_Internal_Unique_ID) AS cde_3_18
FROM   tdgi_central_base a
JOIN   rafy2025_centralized.TD_SAR_CDE_3_18_2025 b
  ON   a.full_nm = upper(b.STR_LEILAs_Customer_Nam)
""").collect()[0][0]

results['3.18'] = cde_3_18
print(f'CDE 3.18 (STR/SAR): {cde_3_18:>12,}')

---
## Step 7: ABAC Metrics
Most ABAC metrics are hardcoded or use previously cached data.

In [ ]:
# ============================================================
# ABAC Hardcoded Metrics
# ============================================================

# ABAC1.1 — Hardcoded 0 (no non-personal for HR jurisdictions)
results['ABAC1.1'] = 0
print(f'ABAC1.1: {0:>12} (hardcoded — no non-personal for HR jurisdictions)')

# ABAC1.2 — Hardcoded 'Not Available' (SD2 not developed)
results['ABAC1.2'] = 'Not Available'
print(f'ABAC1.2: {"Not Available":>12} (hardcoded — SD2 not developed)')

# ABAC3.1 — Hardcoded 0 (3.2 not available)
results['ABAC3.1'] = 0
print(f'ABAC3.1: {0:>12} (hardcoded — CDE 3.2 not available)')

# ABAC3.2 — Hardcoded 0 (3.10 not available)
results['ABAC3.2'] = 0
print(f'ABAC3.2: {0:>12} (hardcoded — CDE 3.10 not available)')

In [ ]:
# ============================================================
# ABAC1.3 — High-risk industry ratio
# sum(case High) / count(distinct acc_account_no) * 100
# from tdigi_high_risk_industry JOIN Industry_Reference_list
# ============================================================
abac_1_3 = spark.sql("""
SELECT
    ROUND(
        sum(CASE WHEN upper(trim(b.Updated_Risk_Rating)) = 'HIGH' THEN 1 ELSE 0 END)
        * 100.0
        / count(distinct a.acc_account_no),
    2) AS abac_1_3
FROM   ra_fy_2025.tdigi_high_risk_industry a
JOIN   ra_adido_2025.industry_ref_list_ca2025 b
  ON   a.industry_code = b.Industry_Code
""").collect()[0][0]

results['ABAC1.3'] = abac_1_3
print(f'ABAC1.3 (HR Industry %): {abac_1_3:>12}%')

---
## Step 8: Results Summary
All metrics collected and printed in a single table.

In [ ]:
# ============================================================
# Final Results Summary
# ============================================================
print('=' * 65)
print('TDI 101522 TDGI — All Metrics Summary')
print('=' * 65)
print()

# Define display order by section
sections = [
    ('SEGMENT', ['SD1', 'SD3', '1.6', '1.7', 'SD4', 'SD5',
                 '4.1A', '4.1B', '4.1C']),
    ('BRANCH', ['5.1', '5.2', '5.3', '5.4', '5.5', '5.6',
                '5.7', '5.8', '5.9']),
    ('POLICY', ['6.5A', '6.5B']),
    ('TRANSACTION (counts)', ['3.1', '3.2', '3.3', '3.4',
                              '3.5', '3.6', '3.7', '3.8']),
    ('TRANSACTION (amounts)', ['3.9', '3.10', '3.11', '3.12',
                               '3.13', '3.14', '3.15', '3.16']),
    ('CENTRALIZED', ['1.8', '1.9', 'SD6', '3.17', '3.18']),
    ('ABAC', ['ABAC1.1', 'ABAC1.2', 'ABAC1.3', 'ABAC3.1',
              'ABAC3.2', 'ABAC5.1']),
]

for section_name, metrics in sections:
    print(f'\n--- {section_name} ---')
    for m in metrics:
        val = results.get(m, 'N/A')
        if isinstance(val, (int, float)):
            if isinstance(val, float) and val != int(val):
                print(f'  {m:12s} {val:>15,.2f}')
            else:
                print(f'  {m:12s} {val:>15,}')
        else:
            print(f'  {m:12s} {str(val):>15}')

print()
print('=' * 65)
print(f'Total metrics computed: {len(results)}')
print('=' * 65)

---
## Cleanup (Optional)
Uncache temporary views to free cluster memory.

In [ ]:
# ============================================================
# Uncache all temp views
# ============================================================
cached_views = [
    'tdgi_2724', 'tdgi_2725', 'tdgi_2727',
    'tdgi_person', 'tdgi_branches', 'tdgi_2721',
    'tdgi_txn', 'tdgi_central_base',
]
for v in cached_views:
    try:
        spark.sql(f'UNCACHE TABLE {v}')
        print(f'Uncached {v}')
    except:
        pass
print('\nCleanup complete.')